In [1]:
import pandas as pd
import numpy as np

maximize the macro F1 SCORE, balance the precision and recall, and find the optimal thresholds for categorizing data into 'H', 'M', and 'L' based on a score column and true labels.

In [7]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Kano/Data/raw/'
data_temp = '../Kano/Data/processed/'
model_outputs = '../Kano/Data/outputs/'

In [12]:
df = pd.read_csv('/Users/moriseiitsu/Dropbox/IDEAMAPS/ideamaps-agile-paper/ideamaps-healtcare-access-dep-agile/Kano/Data/raw/2026-03-11T22_05_20+00_00_urh.csv')
print(df.columns)


Index(['id', 'created_at', 'validation', 'user_id', 'output_id',
       'output_model', 'output_model_city_name', 'output_model_city_country',
       'output_model_subdomain_name', 'output_result', 'output_latitude',
       'output_longitude'],
      dtype='str')


In [ ]:
# spcial join accessibility index to the dataset
accessibility_index = pd.read_csv('')

In [ ]:
# ==========================================
# 1. THE OPTIMIZATION ALGORITHM
# ==========================================
def optimize_thresholds(df, score_col='score', validation='true_label'):
    """
    Finds the optimal x and y thresholds to maximize matching categories.
    Assumes: 0 to x is 'H', x to y is 'M', and y to 1 is 'L'.
    """
    # Get all unique, sorted scores to test as potential cut-offs
    candidates = np.sort(df[score_col].unique())
    
    # Extract as numpy arrays for much faster vector calculations
    scores = df[score_col].values
    true_labels = df[label_col].values
    total_samples = len(df)
    
    best_x = 0
    best_y = 0
    max_matches = -1
    
    # Iterate through all valid pairs of x and y (where x < y)
    for i in range(len(candidates)):
        x = candidates[i]
        
        # y must be greater than x
        for j in range(i + 1, len(candidates)):
            y = candidates[j]
            
            # Generate predictions based on current x and y
            # Condition: <= x is 'H', <= y is 'M', else 'L'
            predictions = np.where(scores <= x, 'H',
                                   np.where(scores <= y, 'M', 'L'))
            
            # Count how many predictions match the human validation
            matches = np.sum(predictions == true_labels)
            
            # Update best parameters if this combination is better
            if matches > max_matches:
                max_matches = matches
                best_x = x
                best_y = y
 
    # Calculate final accuracy percentage
    accuracy = (max_matches / total_samples) * 100
                
    return best_x, best_y, max_matches, accuracy
 
# ==========================================
# 2. GENERATE MOCK DATA (For testing)
# ==========================================
# Let's create a fake dataset of 500 records to prove it works
np.random.seed(42)
mock_scores = np.random.uniform(0, 1, 500)
 
# We will assign true labels with a simulated "perfect" split of x=0.35 and y=0.75,
# but add some random noise so it's not a trivially perfect dataset.
mock_labels = []
for s in mock_scores:
    # Adding a tiny bit of random noise to the score for realistic overlap
    noisy_s = s + np.random.normal(0, 0.1)
    if noisy_s <= 0.35:
        mock_labels.append('H')
    elif noisy_s <= 0.75:
        mock_labels.append('M')
    else:
        mock_labels.append('L')
 
# Create the DataFrame
df = pd.DataFrame({'accessibility_score': mock_scores, 'human_label': mock_labels})
print(f"Dataset preview:\n{df.head()}\n")
 
# ==========================================
# 3. RUN THE OPTIMIZER
# ==========================================
print("Running optimization... (this takes less than a second)")
optimal_x, optimal_y, total_matches, accuracy_pct = optimize_thresholds(
    df=df,
    score_col='accessibility_score',
    label_col='human_label'
)
 
# Output the results
print("-" * 30)
print("OPTIMIZATION RESULTS")
print("-" * 30)
print(f"Best 'x' threshold (H / M split): {optimal_x:.4f}")
print(f"Best 'y' threshold (M / L split): {optimal_y:.4f}")
print(f"Total Correct Matches: {total_matches} out of {len(df)}")
print(f"Overall Accuracy: {accuracy_pct:.2f}%")